In [ ]:
## detecting the local yaml files
# change from V1 to V2: saves the action yamls in a subfolder "actions".

In [ ]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import csv
import os
import re
import shutil
import stat
import subprocess
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Optional, Set
from urllib.parse import urlparse

import pandas as pd
import requests
from dotenv import load_dotenv

# ========= CONFIG & PATHS =========

# Your env file location (as you stated)
ENV_DIR = Path(r"C:\GitHub\Android-Mobile-Apps")
ENV_FILE_NAME = "All_Tokens.env"  # keep this exact filename

# Cutoff date (ISO 8601, UTC)
CUTOFF_ISO = "2025-08-10T23:59:59Z"

# Base RQ1 directory
BASE_RQ1 = Path(r"D:")

# URL list for all repos
URL_LIST_CSV = Path(r"D:\URL_List.csv")

# Local instrumentation detection results
LOCAL_INSTRU_DIR = BASE_RQ1 / "Local_Instru_Tests"
LOCAL_NON_WORKFLOW_CSV = LOCAL_INSTRU_DIR / "Repos_with_Local_GHA_Actions_NON_WORKFLOW.csv"

# TEMP git dirs per repo (no checkout)
CLONE_DIR = LOCAL_INSTRU_DIR / "Local_Cloned_Repos"

# === NEW: where we store extracted local action YAML files for parsing ===
# Layout: D:\All_Action_YMLs\<owner_repo_key>\<relative_path_inside_repo>
# Example: owner_repo_key = "owner.repo", relative_path = ".github/actions/my-action/action.yml"
# => D:\All_Action_YMLs\owner.repo\.github\actions\my-action\action.yml
OUTPUT_YML_DIR = Path(r"D:\All_Action_YMLs")

# Index CSV describing extracted local YAMLs (kept under Local_Instru_Tests)
LOCAL_INDEX_CSV = LOCAL_INSTRU_DIR / "Local_YML_Extraction_Index.csv"

# Whether to keep temp git dirs after extraction
KEEP_CLONES = False

# --- GitHub API behavior ---
REQUEST_TIMEOUT_SECONDS = 30

# If tokens are rate-limited, do you want to WAIT until reset?
# For your use-case, default False so we just fall back to HEAD and continue extraction.
WAIT_FOR_RATE_LIMIT_RESET = False

# Secondary limit backoff (short, bounded)
SECONDARY_BACKOFF_BASE_SECONDS = 10
SECONDARY_BACKOFF_MAX_SECONDS = 90
MAX_SECONDARY_RETRIES_PER_CALL = 3
MAX_TOTAL_API_ATTEMPTS_PER_CALL = 50

# Git on Windows
GIT_PREFIX = ["git", "-c", "core.longpaths=true"]


# ========= subprocess helpers =========

def run_text(cmd, cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    """Run a command and capture stdout+stderr as text, safely decoded."""
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=check,
    )

def run_bytes(cmd, cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    """Run a command and capture stdout+stderr as bytes."""
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=False,
        check=check,
    )

def force_remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)
    func(path)

def sanitize_token(s: str) -> str:
    return re.sub(r"[^a-z0-9._+-]", "_", s.lower())

def split_stem_and_suffixes(name: str):
    p = Path(name)
    suffixes = "".join(p.suffixes)
    if suffixes:
        return name[:-len(suffixes)], suffixes
    return name, ""

def _counter_candidate(base_name: str, n: int) -> str:
    if n == 1:
        return base_name
    if "++" in base_name:
        head, tail = base_name.split("++", 1)
        tail_stem, tail_suf = split_stem_and_suffixes(tail)
        return f"{head}++{tail_stem}__{n}{tail_suf}"
    stem, suf = split_stem_and_suffixes(base_name)
    return f"{stem}__{n}{suf}"

def resolve_counter_name(bucket_dir: Path, base_name: str, in_memory_taken: Set[str]) -> str:
    n = 1
    while True:
        candidate = _counter_candidate(base_name, n)
        if candidate not in in_memory_taken and not (bucket_dir / candidate).exists():
            in_memory_taken.add(candidate)
            return candidate
        n += 1

def make_flat_filename(owner: str, project: str, ci_platform: str, file_name: str) -> str:
    owner_tok = sanitize_token(owner)
    project_tok = sanitize_token(project)
    ci_tok = sanitize_token(ci_platform or "other")
    file_lower = file_name.lower()
    return f"{owner_tok}.{project_tok}__{ci_tok}++{file_lower}"

def save_bytes(bucket_dir: Path, flat_filename: str, content: bytes, in_memory_taken: Set[str]) -> Path:
    bucket_dir.mkdir(parents=True, exist_ok=True)
    final_name = resolve_counter_name(bucket_dir, flat_filename, in_memory_taken)
    dest = bucket_dir / final_name
    dest.write_bytes(content)
    return dest


# ========= branch detection =========

def detect_default_branch(repo_url: str) -> str:
    try:
        out = run_text(GIT_PREFIX + ["ls-remote", "--symref", repo_url, "HEAD"]).stdout
        for line in out.splitlines():
            s = line.strip()
            if s.startswith("ref: ") and s.endswith("HEAD"):
                ref = s.split()[1]  # refs/heads/main
                if ref.startswith("refs/heads/"):
                    return ref.split("/", 2)[2]
    except Exception:
        pass

    for guess in ("main", "master"):
        try:
            run_text(GIT_PREFIX + ["ls-remote", repo_url, f"refs/heads/{guess}"], check=True)
            return guess
        except Exception:
            continue

    raise RuntimeError("Could not determine default branch via git ls-remote")


# ========= load tokens (notebook + script safe) =========

def find_env_file(env_dir: Path, env_filename: str) -> Path:
    """
    Try these locations (in order):
      1) ENV_DIR/env_filename (your stated location)
      2) cwd/env_filename
      3) BASE_RQ1/env_filename
      4) parents of cwd (up to 5 levels)
      5) script directory (if available)
    """
    candidates: list[Path] = []

    candidates.append(env_dir / env_filename)
    candidates.append(Path.cwd() / env_filename)
    candidates.append(BASE_RQ1 / env_filename)

    # walk up cwd a bit (helpful in notebooks)
    cur = Path.cwd()
    for _ in range(5):
        candidates.append(cur / env_filename)
        if cur.parent == cur:
            break
        cur = cur.parent

    # script dir (if running as a script)
    try:
        script_dir = Path(__file__).resolve().parent  # type: ignore[name-defined]
        candidates.append(script_dir / env_filename)
    except NameError:
        pass

    env_path = next((p for p in candidates if p.exists()), None)
    if env_path is None:
        raise FileNotFoundError(
            f"Could not find env file '{env_filename}'. Tried:\n" +
            "\n".join(f"  {p}" for p in candidates)
        )
    return env_path

def load_tokens(env_dir: Path, env_filename: str) -> list[str]:
    env_path = find_env_file(env_dir, env_filename)
    load_dotenv(dotenv_path=str(env_path), override=True)

    tokens: list[str] = []
    for i in range(1, 50):
        t = os.getenv(f"GITHUB_TOKEN_{i}")
        if t:
            tokens.append(t.strip())
    return tokens


TOKENS = load_tokens(ENV_DIR, ENV_FILE_NAME)


# ========= GitHub API: rotation + backoff =========

GITHUB_API_BASE_HEADERS = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

@dataclass
class TokenState:
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None
    disabled: bool = False

@dataclass
class TokenPool:
    tokens: list[str]
    idx: int = 0
    state: Dict[str, TokenState] = field(default_factory=dict)

    def __post_init__(self):
        for t in self.tokens:
            self.state.setdefault(t, TokenState())

    def _is_usable(self, token: str) -> bool:
        st = self.state[token]
        if st.disabled:
            return False
        now = int(time.time())
        if st.remaining == 0 and st.reset_epoch and st.reset_epoch > now:
            return False
        return True

    def next_token(self) -> Optional[str]:
        if not self.tokens:
            return None
        for _ in range(len(self.tokens)):
            token = self.tokens[self.idx % len(self.tokens)]
            self.idx += 1
            if self._is_usable(token):
                return token
        return None

    def mark_rate_info(self, token: str, remaining: Optional[int], reset_epoch: Optional[int]) -> None:
        st = self.state[token]
        if remaining is not None:
            st.remaining = remaining
        if reset_epoch is not None:
            st.reset_epoch = reset_epoch

    def disable(self, token: str) -> None:
        self.state[token].disabled = True

    def earliest_reset(self) -> Optional[int]:
        resets = [
            st.reset_epoch
            for st in self.state.values()
            if st.reset_epoch is not None and st.remaining == 0 and not st.disabled
        ]
        return min(resets) if resets else None


SESSION = requests.Session()
POOL = TokenPool(TOKENS)

def _parse_int_header(resp: requests.Response, name: str) -> Optional[int]:
    v = resp.headers.get(name)
    if v is None:
        return None
    try:
        return int(v)
    except Exception:
        return None

def github_get(url: str, params: dict) -> requests.Response:
    secondary_tries = 0
    attempts = 0

    while attempts < MAX_TOTAL_API_ATTEMPTS_PER_CALL:
        attempts += 1

        token = POOL.next_token()
        headers = dict(GITHUB_API_BASE_HEADERS)
        if token:
            headers["Authorization"] = f"Bearer {token}"

        resp = SESSION.get(url, headers=headers, params=params, timeout=REQUEST_TIMEOUT_SECONDS)

        # Parse message if possible
        msg = ""
        try:
            j = resp.json()
            if isinstance(j, dict):
                msg = str(j.get("message", "") or "")
        except Exception:
            pass

        remaining = _parse_int_header(resp, "X-RateLimit-Remaining")
        reset = _parse_int_header(resp, "X-RateLimit-Reset")
        retry_after = _parse_int_header(resp, "Retry-After")

        if token:
            POOL.mark_rate_info(token, remaining, reset)

        if resp.status_code == 200:
            return resp

        if resp.status_code == 401:
            if token:
                print("  GitHub API: 401 Unauthorized; disabling that token.")
                POOL.disable(token)
            return resp

        if resp.status_code in (403, 429):
            low_msg = msg.lower()

            # secondary rate limit
            if "secondary rate limit" in low_msg:
                if secondary_tries >= MAX_SECONDARY_RETRIES_PER_CALL:
                    return resp
                wait_s = retry_after if retry_after is not None else (SECONDARY_BACKOFF_BASE_SECONDS * (2 ** secondary_tries))
                wait_s = min(wait_s, SECONDARY_BACKOFF_MAX_SECONDS)
                secondary_tries += 1
                print(f"  GitHub API: secondary rate limit; sleeping {wait_s}s then retrying...")
                time.sleep(wait_s)
                continue

            # primary rate limit exhausted
            if remaining == 0:
                # try other token
                next_tok = POOL.next_token()
                if next_tok is not None:
                    continue

                # all tokens exhausted
                if not WAIT_FOR_RATE_LIMIT_RESET:
                    return resp

                earliest = POOL.earliest_reset()
                if earliest is None:
                    return resp
                now = int(time.time())
                sleep_s = max(1, earliest - now)
                print(f"  GitHub API: all tokens exhausted; sleeping {sleep_s}s until reset...")
                time.sleep(sleep_s)
                continue

            return resp

        return resp

    return resp

def get_cutoff_commit(owner: str, repo: str, branch: str, cutoff_iso: str) -> Optional[str]:
    url = f"https://api.github.com/repos/{owner}/{repo}/commits"
    params = {"sha": branch, "until": cutoff_iso, "per_page": 1}

    resp = github_get(url, params=params)
    if resp.status_code != 200:
        try:
            j = resp.json()
            msg = j.get("message", "") if isinstance(j, dict) else ""
        except Exception:
            msg = (resp.text or "")[:200]

        print(
            f"  GitHub API error for {owner}/{repo} commits: {resp.status_code} {msg} "
            f"(remaining={resp.headers.get('X-RateLimit-Remaining')}, reset={resp.headers.get('X-RateLimit-Reset')}, "
            f"retry_after={resp.headers.get('Retry-After')})"
        )
        return None

    data = resp.json()
    if not isinstance(data, list) or not data:
        return None
    return data[0].get("sha")


# ========= git read without checkout =========

def normalize_uses_path(uses_path: str) -> str:
    norm = uses_path.strip().strip('"').strip("'").replace("\\", "/")
    if norm.startswith("./"):
        norm = norm[2:]
    norm = norm.lstrip("/")
    norm = norm.rstrip("/")
    if norm in ("", "."):
        return ""  # repo root
    return norm

def git_path_exists(repo_dir: Path, commit_sha: str, rel_path: str) -> bool:
    if not rel_path:
        return False
    target = f"{commit_sha}:{rel_path}"
    proc = run_text(GIT_PREFIX + ["cat-file", "-e", target], cwd=repo_dir, check=False)
    return proc.returncode == 0

def git_show_file(repo_dir: Path, commit_sha: str, rel_path: str) -> Optional[bytes]:
    if not rel_path:
        return None
    target = f"{commit_sha}:{rel_path}"
    proc = run_bytes(GIT_PREFIX + ["show", target], cwd=repo_dir, check=False)
    if proc.returncode != 0:
        return None
    return proc.stdout

def resolve_action_yaml_path_at_commit(repo_dir: Path, commit_sha: str, uses_path: str) -> Optional[str]:
    norm = normalize_uses_path(uses_path)

    # direct YAML file
    if norm.lower().endswith((".yml", ".yaml")):
        return norm if git_path_exists(repo_dir, commit_sha, norm) else None

    # directory action.yml/action.yaml or root action.yml/action.yaml
    for fname in ("action.yml", "action.yaml"):
        candidate = f"{norm}/{fname}" if norm else fname
        if git_path_exists(repo_dir, commit_sha, candidate):
            return candidate

    return None

def prepare_git_dir(repo_git_dir: Path, github_url: str) -> None:
    if repo_git_dir.exists():
        shutil.rmtree(repo_git_dir, onerror=force_remove_readonly)
    repo_git_dir.mkdir(parents=True, exist_ok=True)
    run_text(GIT_PREFIX + ["init"], cwd=repo_git_dir)
    run_text(GIT_PREFIX + ["remote", "add", "origin", github_url], cwd=repo_git_dir)

def fetch_ref(repo_git_dir: Path, ref: str) -> str:
    run_text(GIT_PREFIX + ["fetch", "--depth", "1", "origin", ref], cwd=repo_git_dir)
    sha = run_text(GIT_PREFIX + ["rev-parse", "FETCH_HEAD"], cwd=repo_git_dir).stdout.strip()
    return sha


# ========= MAIN =========

def main():
    LOCAL_INSTRU_DIR.mkdir(parents=True, exist_ok=True)
    CLONE_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_YML_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Loaded {len(TOKENS)} GitHub token(s) from {ENV_DIR}\\{ENV_FILE_NAME}")
    if len(TOKENS) == 0:
        print("WARNING: Running unauthenticated -> you WILL hit API rate limits quickly.")

    # 1) Load URL list
    if not URL_LIST_CSV.exists():
        raise FileNotFoundError(f"URL list CSV not found: {URL_LIST_CSV}")

    url_df = pd.read_csv(URL_LIST_CSV)
    url_df.columns = url_df.columns.str.strip().str.lower()
    if "github_url" not in url_df.columns:
        raise ValueError("URL_List.csv must contain a 'github_url' column")

    owner_repo_to_info: Dict[str, Dict[str, str]] = {}
    for raw_url in url_df["github_url"].dropna().astype(str):
        raw_url = raw_url.strip()
        if not raw_url.startswith("http"):
            continue
        parsed = urlparse(raw_url)
        parts = parsed.path.strip("/").split("/")
        if len(parts) < 2:
            continue
        owner_raw, project_raw = parts[0], parts[1].replace(".git", "")
        key = f"{sanitize_token(owner_raw)}.{sanitize_token(project_raw)}"
        owner_repo_to_info[key] = {"github_url": raw_url, "owner": owner_raw, "project": project_raw}

    print(f"Built mapping for {len(owner_repo_to_info)} repos from URL_List.csv")

    # 2) Load NON_WORKFLOW local uses CSV
    if not LOCAL_NON_WORKFLOW_CSV.exists():
        raise FileNotFoundError(f"Local NON_WORKFLOW CSV not found: {LOCAL_NON_WORKFLOW_CSV}")

    local_df = pd.read_csv(LOCAL_NON_WORKFLOW_CSV)
    local_df.columns = local_df.columns.str.strip()

    if "owner_repo" not in local_df.columns or "uses_paths" not in local_df.columns:
        raise ValueError("Repos_with_Local_GHA_Actions_NON_WORKFLOW.csv must contain 'owner_repo' and 'uses_paths' columns")

    local_uses_by_repo: Dict[str, Set[str]] = {}
    for _, row in local_df.iterrows():
        owner_repo = str(row["owner_repo"]).strip()
        uses_field = str(row["uses_paths"]).strip()
        if not owner_repo or not uses_field or uses_field.lower() == "nan":
            continue
        for raw_part in uses_field.split(";"):
            up = raw_part.strip()
            if up:
                local_uses_by_repo.setdefault(owner_repo, set()).add(up)

    print(f"Found {len(local_uses_by_repo)} repos with NON-workflow local uses")

    # 3) Prepare index CSV
    index_fields = [
        "owner",
        "repo",
        "owner_repo_key",
        "github_url",
        "default_branch",
        "snapshot_commit",   # actual fetched SHA (cutoff or HEAD)
        "local_use_path",
        "relative_path",
        "filename",
        "flat_filename",     # now used to store relative_path (repo-relative)
        "saved_to",
    ]
    if not LOCAL_INDEX_CSV.exists():
        with LOCAL_INDEX_CSV.open("w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=index_fields).writeheader()

    # 4) Process repos
    for owner_repo_key, uses_set in sorted(local_uses_by_repo.items()):
        info = owner_repo_to_info.get(owner_repo_key)
        if not info:
            print(f"No URL found in URL_List.csv for key: {owner_repo_key} – skipping.")
            continue

        github_url = info["github_url"]
        owner = info["owner"]
        project = info["project"]

        print(f"\nProcessing repo {owner_repo_key} -> {github_url}")

        # default branch
        try:
            default_branch = detect_default_branch(github_url)
            print(f"  Default branch: {default_branch}")
        except Exception as e:
            print(f"  Failed to detect default branch: {e}")
            continue

        # cutoff commit via API; if none or API fails, fall back to HEAD branch
        cutoff_sha = get_cutoff_commit(owner, project, default_branch, CUTOFF_ISO)
        fetch_ref_name = cutoff_sha or default_branch

        if cutoff_sha:
            print(f"  Cutoff commit (API): {cutoff_sha}")
        else:
            print(f"  No cutoff commit (none before cutoff OR API failure). Falling back to HEAD of {default_branch}")

        # temp git dir
        repo_git_dir = CLONE_DIR / owner_repo_key.replace("/", "_")
        try:
            prepare_git_dir(repo_git_dir, github_url)
            snapshot_sha = fetch_ref(repo_git_dir, fetch_ref_name)
            print(f"  Fetched snapshot commit: {snapshot_sha}")
        except Exception as e:
            if isinstance(e, subprocess.CalledProcessError):
                print(f"  Failed to prepare/fetch repo:\n{e.stdout}")
            else:
                print(f"  Failed to prepare/fetch repo: {e}")
            continue

        # resolve & save each local use path
        for local_path in sorted(uses_set):
            print(f"    Resolving local use path: {local_path}")

            rel_file_path = resolve_action_yaml_path_at_commit(repo_git_dir, snapshot_sha, local_path)
            if not rel_file_path:
                print(f"    Could not resolve YAML for local path in snapshot: {local_path}")
                continue

            content = git_show_file(repo_git_dir, snapshot_sha, rel_file_path)
            if content is None:
                print(f"    Could not read file content via git show: {rel_file_path}")
                continue

            filename = Path(rel_file_path).name

            # === SAVE SCHEME (NEW) ===
            # Save under: D:\All_Action_YMLs\<owner_repo_key>\<relative_path_inside_repo>
            # Example: owner_repo_key = "owner.repo",
            #          rel_file_path = ".github/actions/my-action/action.yml"
            # => D:\All_Action_YMLs\owner.repo\.github\actions\my-action\action.yml
            dest_path = OUTPUT_YML_DIR / owner_repo_key / rel_file_path
            dest_path.parent.mkdir(parents=True, exist_ok=True)
            dest_path.write_bytes(content)

            with LOCAL_INDEX_CSV.open("a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=index_fields)
                writer.writerow({
                    "owner": owner,
                    "repo": project,
                    "owner_repo_key": owner_repo_key,
                    "github_url": github_url,
                    "default_branch": default_branch,
                    "snapshot_commit": snapshot_sha,
                    "local_use_path": local_path,
                    "relative_path": rel_file_path,
                    "filename": filename,
                    "flat_filename": rel_file_path,   # repo-relative path
                    "saved_to": str(dest_path),
                })

            print(f"    Saved {rel_file_path} -> {dest_path}")

        if not KEEP_CLONES:
            try:
                shutil.rmtree(repo_git_dir, onerror=force_remove_readonly)
            except Exception as e:
                print(f"  Failed to delete temp git dir {repo_git_dir}: {e}")

    print("\nDone. Local action YAMLs stored in:")
    print(f"  {OUTPUT_YML_DIR}")
    print("Index CSV:")
    print(f"  {LOCAL_INDEX_CSV}")


if __name__ == "__main__":
    main()


In [ ]:
## Download the local yml files through a similar clone with the initial data extraction with the cut off date: Aug 10, 2025

In [ ]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import csv
import os
import re
import shutil
import stat
import subprocess
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Optional, Set
from urllib.parse import urlparse

import pandas as pd
import requests
from dotenv import load_dotenv

# ========= CONFIG & PATHS =========

# Your env file location
ENV_DIR = Path(r"C:\GitHub\Android-Mobile-Apps")
ENV_FILE_NAME = "All_Tokens.env"  # keep this exact filename

# Cutoff date (ISO 8601, UTC)
CUTOFF_ISO = "2025-08-10T23:59:59Z"

# Base directory (drive root)
BASE_RQ1 = Path(r"D:\temp")  # <--- key: we treat D:\ as base

# URL list for all repos
URL_LIST_CSV = BASE_RQ1 / "URL_List.csv"

# Local instrumentation detection results (where your NON_WORKFLOW CSV lives)
LOCAL_INSTRU_DIR = BASE_RQ1 / "Local_Instru_Tests"
LOCAL_NON_WORKFLOW_CSV = LOCAL_INSTRU_DIR / "Repos_with_Local_GHA_Actions_NON_WORKFLOW.csv"

# TEMP git dirs per repo (no checkout)
CLONE_DIR = LOCAL_INSTRU_DIR / "Local_Cloned_Repos"

# Where we store extracted local YAML files (ACTION.YML/ACTION.YAML),
# preserving repo-relative paths:
#   D:\All_Action_YMLs\<owner.repo>\<relative_path_inside_repo>
OUTPUT_YML_DIR = BASE_RQ1 / "All_Action_YMLs"

# Index CSV describing extracted local YAMLs
LOCAL_INDEX_CSV = LOCAL_INSTRU_DIR / "Local_YML_Extraction_Index.csv"

# Whether to keep temp git dirs after extraction
KEEP_CLONES = False

# --- GitHub API behavior ---
REQUEST_TIMEOUT_SECONDS = 30

# If tokens are rate-limited, do you want to WAIT until reset?
WAIT_FOR_RATE_LIMIT_RESET = False

# Secondary limit backoff (short, bounded)
SECONDARY_BACKOFF_BASE_SECONDS = 10
SECONDARY_BACKOFF_MAX_SECONDS = 90
MAX_SECONDARY_RETRIES_PER_CALL = 3
MAX_TOTAL_API_ATTEMPTS_PER_CALL = 50

# Git on Windows
GIT_PREFIX = ["git", "-c", "core.longpaths=true"]


# ========= subprocess helpers =========

def run_text(cmd, cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    """Run a command and capture stdout+stderr as text, safely decoded."""
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=check,
    )


def run_bytes(cmd, cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    """Run a command and capture stdout+stderr as bytes."""
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=False,
        check=check,
    )


def force_remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)
    func(path)


def sanitize_token(s: str) -> str:
    return re.sub(r"[^a-z0-9._+-]", "_", s.lower())


# ========= branch detection =========

def detect_default_branch(repo_url: str) -> str:
    try:
        out = run_text(GIT_PREFIX + ["ls-remote", "--symref", repo_url, "HEAD"]).stdout
        for line in out.splitlines():
            s = line.strip()
            if s.startswith("ref: ") and s.endswith("HEAD"):
                ref = s.split()[1]  # refs/heads/main
                if ref.startswith("refs/heads/"):
                    return ref.split("/", 2)[2]
    except Exception:
        pass

    # fallback guesses
    for guess in ("main", "master"):
        try:
            run_text(GIT_PREFIX + ["ls-remote", repo_url, f"refs/heads/{guess}"], check=True)
            return guess
        except Exception:
            continue

    raise RuntimeError("Could not determine default branch via git ls-remote")


# ========= load tokens (notebook + script safe) =========

def find_env_file(env_dir: Path, env_filename: str) -> Path:
    """
    Try these locations (in order):
      1) ENV_DIR/env_filename (your stated location)
      2) cwd/env_filename
      3) BASE_RQ1/env_filename
      4) parents of cwd (up to 5 levels)
      5) script directory (if available)
    """
    candidates: list[Path] = []

    candidates.append(env_dir / env_filename)
    candidates.append(Path.cwd() / env_filename)
    candidates.append(BASE_RQ1 / env_filename)

    # walk up cwd a bit (helpful in notebooks)
    cur = Path.cwd()
    for _ in range(5):
        candidates.append(cur / env_filename)
        if cur.parent == cur:
            break
        cur = cur.parent

    # script dir (if running as a script)
    try:
        script_dir = Path(__file__).resolve().parent  # type: ignore[name-defined]
        candidates.append(script_dir / env_filename)
    except NameError:
        pass

    env_path = next((p for p in candidates if p.exists()), None)
    if env_path is None:
        raise FileNotFoundError(
            f"Could not find env file '{env_filename}'. Tried:\n" +
            "\n".join(f"  {p}" for p in candidates)
        )
    return env_path


def load_tokens(env_dir: Path, env_filename: str) -> list[str]:
    env_path = find_env_file(env_dir, env_filename)
    load_dotenv(dotenv_path=str(env_path), override=True)

    tokens: list[str] = []
    for i in range(1, 50):
        t = os.getenv(f"GITHUB_TOKEN_{i}")
        if t:
            tokens.append(t.strip())
    return tokens


TOKENS = load_tokens(ENV_DIR, ENV_FILE_NAME)


# ========= GitHub API: rotation + backoff =========

GITHUB_API_BASE_HEADERS = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}


@dataclass
class TokenState:
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None
    disabled: bool = False


@dataclass
class TokenPool:
    tokens: list[str]
    idx: int = 0
    state: Dict[str, TokenState] = field(default_factory=dict)

    def __post_init__(self):
        for t in self.tokens:
            self.state.setdefault(t, TokenState())

    def _is_usable(self, token: str) -> bool:
        st = self.state[token]
        if st.disabled:
            return False
        now = int(time.time())
        if st.remaining == 0 and st.reset_epoch and st.reset_epoch > now:
            return False
        return True

    def next_token(self) -> Optional[str]:
        if not self.tokens:
            return None
        for _ in range(len(self.tokens)):
            token = self.tokens[self.idx % len(self.tokens)]
            self.idx += 1
            if self._is_usable(token):
                return token
        return None

    def mark_rate_info(self, token: str, remaining: Optional[int], reset_epoch: Optional[int]) -> None:
        st = self.state[token]
        if remaining is not None:
            st.remaining = remaining
        if reset_epoch is not None:
            st.reset_epoch = reset_epoch

    def disable(self, token: str) -> None:
        self.state[token].disabled = True

    def earliest_reset(self) -> Optional[int]:
        resets = [
            st.reset_epoch
            for st in self.state.values()
            if st.reset_epoch is not None and st.remaining == 0 and not st.disabled
        ]
        return min(resets) if resets else None


SESSION = requests.Session()
POOL = TokenPool(TOKENS)


def _parse_int_header(resp: requests.Response, name: str) -> Optional[int]:
    v = resp.headers.get(name)
    if v is None:
        return None
    try:
        return int(v)
    except Exception:
        return None


def github_get(url: str, params: dict) -> requests.Response:
    secondary_tries = 0
    attempts = 0

    while attempts < MAX_TOTAL_API_ATTEMPTS_PER_CALL:
        attempts += 1

        token = POOL.next_token()
        headers = dict(GITHUB_API_BASE_HEADERS)
        if token:
            headers["Authorization"] = f"Bearer {token}"

        resp = SESSION.get(url, headers=headers, params=params, timeout=REQUEST_TIMEOUT_SECONDS)

        # Parse message if possible
        msg = ""
        try:
            j = resp.json()
            if isinstance(j, dict):
                msg = str(j.get("message", "") or "")
        except Exception:
            pass

        remaining = _parse_int_header(resp, "X-RateLimit-Remaining")
        reset = _parse_int_header(resp, "X-RateLimit-Reset")
        retry_after = _parse_int_header(resp, "Retry-After")

        if token:
            POOL.mark_rate_info(token, remaining, reset)

        if resp.status_code == 200:
            return resp

        if resp.status_code == 401:
            if token:
                print("  GitHub API: 401 Unauthorized; disabling that token.")
                POOL.disable(token)
            return resp

        if resp.status_code in (403, 429):
            low_msg = msg.lower()

            # secondary rate limit
            if "secondary rate limit" in low_msg:
                if secondary_tries >= MAX_SECONDARY_RETRIES_PER_CALL:
                    return resp
                wait_s = retry_after if retry_after is not None else (
                    SECONDARY_BACKOFF_BASE_SECONDS * (2 ** secondary_tries)
                )
                wait_s = min(wait_s, SECONDARY_BACKOFF_MAX_SECONDS)
                secondary_tries += 1
                print(f"  GitHub API: secondary rate limit; sleeping {wait_s}s then retrying...")
                time.sleep(wait_s)
                continue

            # primary rate limit exhausted
            if remaining == 0:
                # try other token
                next_tok = POOL.next_token()
                if next_tok is not None:
                    continue

                # all tokens exhausted
                if not WAIT_FOR_RATE_LIMIT_RESET:
                    return resp

                earliest = POOL.earliest_reset()
                if earliest is None:
                    return resp
                now = int(time.time())
                sleep_s = max(1, earliest - now)
                print(f"  GitHub API: all tokens exhausted; sleeping {sleep_s}s until reset...")
                time.sleep(sleep_s)
                continue

            return resp

        return resp

    return resp


def get_cutoff_commit(owner: str, repo: str, branch: str, cutoff_iso: str) -> Optional[str]:
    url = f"https://api.github.com/repos/{owner}/{repo}/commits"
    params = {"sha": branch, "until": cutoff_iso, "per_page": 1}

    resp = github_get(url, params=params)
    if resp.status_code != 200:
        try:
            j = resp.json()
            msg = j.get("message", "") if isinstance(j, dict) else ""
        except Exception:
            msg = (resp.text or "")[:200]

        print(
            f"  GitHub API error for {owner}/{repo} commits: {resp.status_code} {msg} "
            f"(remaining={resp.headers.get('X-RateLimit-Remaining')}, "
            f"reset={resp.headers.get('X-RateLimit-Reset')}, "
            f"retry_after={resp.headers.get('Retry-After')})"
        )
        return None

    data = resp.json()
    if not isinstance(data, list) or not data:
        return None
    return data[0].get("sha")


# ========= git read without checkout =========

def normalize_uses_path(uses_path: str) -> str:
    norm = uses_path.strip().strip('"').strip("'").replace("\\", "/")
    if norm.startswith("./"):
        norm = norm[2:]
    norm = norm.lstrip("/")
    norm = norm.rstrip("/")
    if norm in ("", "."):
        return ""  # repo root
    return norm


def git_path_exists(repo_dir: Path, commit_sha: str, rel_path: str) -> bool:
    if not rel_path:
        return False
    target = f"{commit_sha}:{rel_path}"
    proc = run_text(GIT_PREFIX + ["cat-file", "-e", target], cwd=repo_dir, check=False)
    return proc.returncode == 0


def git_show_file(repo_dir: Path, commit_sha: str, rel_path: str) -> Optional[bytes]:
    if not rel_path:
        return None
    target = f"{commit_sha}:{rel_path}"
    proc = run_bytes(GIT_PREFIX + ["show", target], cwd=repo_dir, check=False)
    if proc.returncode != 0:
        return None
    return proc.stdout


def resolve_action_yaml_path_at_commit(
    repo_dir: Path,
    commit_sha: str,
    uses_path: str
) -> Optional[str]:
    """
    Given a 'uses:' path (local to repo), return the repo-relative path
    of the actual action YAML at a specific commit, or None if not found.
    """
    norm = normalize_uses_path(uses_path)

    # direct YAML file
    if norm.lower().endswith((".yml", ".yaml")):
        return norm if git_path_exists(repo_dir, commit_sha, norm) else None

    # directory action.yml/action.yaml or root action.yml/action.yaml
    for fname in ("action.yml", "action.yaml"):
        candidate = f"{norm}/{fname}" if norm else fname
        if git_path_exists(repo_dir, commit_sha, candidate):
            return candidate

    return None


def prepare_git_dir(repo_git_dir: Path, github_url: str) -> None:
    if repo_git_dir.exists():
        shutil.rmtree(repo_git_dir, onerror=force_remove_readonly)
    repo_git_dir.mkdir(parents=True, exist_ok=True)
    run_text(GIT_PREFIX + ["init"], cwd=repo_git_dir)
    run_text(GIT_PREFIX + ["remote", "add", "origin", github_url], cwd=repo_git_dir)


def fetch_ref(repo_git_dir: Path, ref: str) -> str:
    run_text(GIT_PREFIX + ["fetch", "--depth", "1", "origin", ref], cwd=repo_git_dir)
    sha = run_text(GIT_PREFIX + ["rev-parse", "FETCH_HEAD"], cwd=repo_git_dir).stdout.strip()
    return sha


def save_action_file(owner_repo_key: str, relative_path: str, content: bytes) -> Path:
    """
    Save the action YAML preserving its repo-relative path:

        D:\All_Action_YMLs\<owner_repo_key>\<relative_path_inside_repo>

    Example:
        owner_repo_key = 'facebook.litho'
        relative_path  = '.github/actions/android-tests/action.yml'

    => 'D:\\All_Action_YMLs\\facebook.litho\\.github\\actions\\android-tests\\action.yml'
    """
    rel_norm = relative_path.replace("\\", "/").lstrip("/")
    dest = OUTPUT_YML_DIR / owner_repo_key / rel_norm
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(content)
    return dest


# ========= MAIN =========

def main():
    LOCAL_INSTRU_DIR.mkdir(parents=True, exist_ok=True)
    CLONE_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_YML_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Loaded {len(TOKENS)} GitHub token(s) from {ENV_DIR}\\{ENV_FILE_NAME}")
    if len(TOKENS) == 0:
        print("WARNING: Running unauthenticated -> you WILL hit API rate limits quickly.")

    # 1) Load URL list
    if not URL_LIST_CSV.exists():
        raise FileNotFoundError(f"URL list CSV not found: {URL_LIST_CSV}")

    url_df = pd.read_csv(URL_LIST_CSV)
    url_df.columns = url_df.columns.str.strip().str.lower()
    if "github_url" not in url_df.columns:
        raise ValueError("URL_List.csv must contain a 'github_url' column")

    owner_repo_to_info: Dict[str, Dict[str, str]] = {}
    for raw_url in url_df["github_url"].dropna().astype(str):
        raw_url = raw_url.strip()
        if not raw_url.startswith("http"):
            continue
        parsed = urlparse(raw_url)
        parts = parsed.path.strip("/").split("/")
        if len(parts) < 2:
            continue
        owner_raw, project_raw = parts[0], parts[1].replace(".git", "")
        key = f"{sanitize_token(owner_raw)}.{sanitize_token(project_raw)}"
        owner_repo_to_info[key] = {
            "github_url": raw_url,
            "owner": owner_raw,
            "project": project_raw,
        }

    print(f"Built mapping for {len(owner_repo_to_info)} repos from URL_List.csv")

    # 2) Load NON_WORKFLOW local uses CSV
    if not LOCAL_NON_WORKFLOW_CSV.exists():
        raise FileNotFoundError(f"Local NON_WORKFLOW CSV not found: {LOCAL_NON_WORKFLOW_CSV}")

    local_df = pd.read_csv(LOCAL_NON_WORKFLOW_CSV)
    local_df.columns = local_df.columns.str.strip()

    if "owner_repo" not in local_df.columns or "uses_paths" not in local_df.columns:
        raise ValueError(
            "Repos_with_Local_GHA_Actions_NON_WORKFLOW.csv must contain 'owner_repo' and 'uses_paths' columns"
        )

    local_uses_by_repo: Dict[str, Set[str]] = {}
    for _, row in local_df.iterrows():
        owner_repo = str(row["owner_repo"]).strip()
        uses_field = str(row["uses_paths"]).strip()
        if not owner_repo or not uses_field or uses_field.lower() == "nan":
            continue
        for raw_part in uses_field.split(";"):
            up = raw_part.strip()
            if up:
                local_uses_by_repo.setdefault(owner_repo, set()).add(up)

    print(f"Found {len(local_uses_by_repo)} repos with NON-workflow local uses")

    # 3) Prepare index CSV
    index_fields = [
        "owner",
        "repo",
        "owner_repo_key",
        "github_url",
        "default_branch",
        "snapshot_commit",   # actual fetched SHA (cutoff or HEAD)
        "local_use_path",
        "relative_path",
        "filename",
        "flat_filename",     # here: basename of the saved file
        "saved_to",
    ]
    if not LOCAL_INDEX_CSV.exists():
        with LOCAL_INDEX_CSV.open("w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=index_fields).writeheader()

    # 4) Process repos
    for owner_repo_key, uses_set in sorted(local_uses_by_repo.items()):
        info = owner_repo_to_info.get(owner_repo_key)
        if not info:
            print(f"No URL found in URL_List.csv for key: {owner_repo_key} – skipping.")
            continue

        github_url = info["github_url"]
        owner = info["owner"]
        project = info["project"]

        print(f"\nProcessing repo {owner_repo_key} -> {github_url}")

        # default branch
        try:
            default_branch = detect_default_branch(github_url)
            print(f"  Default branch: {default_branch}")
        except Exception as e:
            print(f"  Failed to detect default branch: {e}")
            continue

        # cutoff commit via API; if none or API fails, fall back to HEAD branch
        cutoff_sha = get_cutoff_commit(owner, project, default_branch, CUTOFF_ISO)
        fetch_ref_name = cutoff_sha or default_branch

        if cutoff_sha:
            print(f"  Cutoff commit (API): {cutoff_sha}")
        else:
            print(f"  No cutoff commit (none before cutoff OR API failure). "
                  f"Falling back to HEAD of {default_branch}")

        # temp git dir
        repo_git_dir = CLONE_DIR / owner_repo_key.replace("/", "_")
        try:
            prepare_git_dir(repo_git_dir, github_url)
            snapshot_sha = fetch_ref(repo_git_dir, fetch_ref_name)
            print(f"  Fetched snapshot commit: {snapshot_sha}")
        except Exception as e:
            if isinstance(e, subprocess.CalledProcessError):
                print(f"  Failed to prepare/fetch repo:\n{e.stdout}")
            else:
                print(f"  Failed to prepare/fetch repo: {e}")
            continue

        # resolve & save each local use path
        for local_path in sorted(uses_set):
            print(f"    Resolving local use path: {local_path}")

            rel_file_path = resolve_action_yaml_path_at_commit(
                repo_git_dir, snapshot_sha, local_path
            )
            if not rel_file_path:
                print(f"    Could not resolve YAML for local path in snapshot: {local_path}")
                continue

            content = git_show_file(repo_git_dir, snapshot_sha, rel_file_path)
            if content is None:
                print(f"    Could not read file content via git show: {rel_file_path}")
                continue

            # Save preserving repo-relative folder structure
            dest_path = save_action_file(owner_repo_key, rel_file_path, content)

            filename = Path(rel_file_path).name

            with LOCAL_INDEX_CSV.open("a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=index_fields)
                writer.writerow({
                    "owner": owner,
                    "repo": project,
                    "owner_repo_key": owner_repo_key,
                    "github_url": github_url,
                    "default_branch": default_branch,
                    "snapshot_commit": snapshot_sha,
                    "local_use_path": local_path,
                    "relative_path": rel_file_path,
                    "filename": filename,
                    "flat_filename": dest_path.name,  # basename of the saved file
                    "saved_to": str(dest_path),
                })

            print(f"    Saved {rel_file_path} as {dest_path}")

        if not KEEP_CLONES:
            try:
                shutil.rmtree(repo_git_dir, onerror=force_remove_readonly)
            except Exception as e:
                print(f"  Failed to delete temp git dir {repo_git_dir}: {e}")

    print("\nDone. Local YAMLs stored in:")
    print(f"  {OUTPUT_YML_DIR}")
    print("Index CSV:")
    print(f"  {LOCAL_INDEX_CSV}")


if __name__ == "__main__":
    main()


Loaded 4 GitHub token(s) from C:\GitHub\Android-Mobile-Apps\All_Tokens.env
Built mapping for 4697 repos from URL_List.csv
Found 160 repos with NON-workflow local uses

Processing repo 1q23lyc45.kitsunemagisk -> https://github.com/1q23lyc45/KitsuneMagisk
  Default branch: kitsune
  Cutoff commit (API): b030f742cbb02c01c54f250555fea16b5389fea8
  Fetched snapshot commit: b030f742cbb02c01c54f250555fea16b5389fea8
    Resolving local use path: ./.github/actions/setup
    Saved .github/actions/setup/action.yml as D:\temp\All_Action_YMLs\1q23lyc45.kitsunemagisk\.github\actions\setup\action.yml

Processing repo 4accccc.vivo-magisk-suu -> https://github.com/4accccc/vivo-Magisk-suu
